# Experiment 2 · 6-fold status overview (read-only)

After mounting Drive, shows the status of all 6 folds at once. **Read-only**: does not start, stop or modify anything; no GPU needed (a CPU runtime is enough and saves compute units).

Status rules (based on the logs and result files on Drive):
- **evaluated**: `eval_test/eval_summary.json` exists
- **trained, awaiting eval**: the log records that training finished, but the evaluation files do not exist -> run (8) in that fold's notebook
- **running**: the log was updated within 5 minutes
- **interrupted, resume needed**: a log exists but has not been updated for over 5 minutes and training has not finished -> in that fold's notebook run (1) (2) (3) (4) (6) as described under "resume after disconnect"
- **not started**: no log

Note: Drive sync lags; logs written by other sessions may show up here 1-2 minutes late.

## (1) Mount Drive and settings

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/Flood2'
OUT_ROOT = f'{DRIVE_ROOT}/models'
MAX_EPOCHS = 50
FOLDS = ['irma', 'matthew', 'ian', 'milton', 'dorian', 'beryl']

## (2) 6-fold status table (rerun this cell to refresh)

In [ ]:
import os, csv, json, time, re
import pandas as pd
from datetime import datetime

def hms_to_s(s):
    h, m, x = (int(v) for v in s.split(':')); return h * 3600 + m * 60 + x

def fmt(sec):
    if sec is None: return ''
    sec = int(sec); return '%d:%02d:%02d' % (sec // 3600, sec % 3600 // 60, sec % 60)

rows = []
for f in FOLDS:
    d = f'{OUT_ROOT}/exp2/fold_{f}'; log = f'{d}/train_stdout.log'; csvp = f'{d}/train_log.csv'
    evp = f'{d}/eval_test/eval_summary.json'; infop = f'{d}/fold_info.json'
    r = {'fold': f}
    if not os.path.exists(log):
        r['status'] = 'not started'; rows.append(r); continue
    txt = open(log, errors='replace').read(); age = time.time() - os.path.getmtime(log)
    ended = ('training finished' in txt) or ('finished previously' in txt)
    tl = list(csv.DictReader(open(csvp))) if os.path.exists(csvp) else []
    secs = [float(x['epoch_sec']) for x in tl]
    prog = re.findall(r'ep (\d+)/(\d+) step (\d+)/(\d+) \| epoch elapsed ([\d:]+) remaining ([\d:]+) \| total elapsed ([\d:]+) total remaining ([\d:]+)', txt)
    if os.path.exists(evp):
        r['status'] = 'evaluated'
    elif ended:
        r['status'] = 'trained, awaiting eval'
    elif age < 300:
        r['status'] = 'running'
    else:
        r['status'] = 'interrupted, resume needed'
    r['epochs done'] = f'{len(tl)} / {MAX_EPOCHS}'
    if r['status'] == 'running' and prog:
        p = prog[-1]; r['current epoch / step'] = f'epoch {p[0]} step {p[2]}/{p[3]}'
    if tl:
        b = min(tl, key=lambda x: float(x['val_loss'])); r['best epoch'] = int(b['epoch']); r['best val'] = '%.3g' % float(b['val_loss'])
    used = sum(secs)
    if r['status'] == 'running' and prog and int(prog[-1][0]) == len(tl) + 1:
        used += hms_to_s(prog[-1][4])
    r['elapsed'] = fmt(used)
    if r['status'] == 'running' and prog:
        r['ETA (all epochs)'] = prog[-1][7]
    r['log updated'] = f'{age / 60:.0f} min ago'
    if os.path.exists(infop):
        s = json.load(open(infop))['sessions']; r['GPU'] = s[-1].get('gpu', s[-1].get('device')); r['sessions'] = len(s)
    if os.path.exists(evp):
        ev = json.load(open(evp)); e = ev['events'][f]
        for m, cn in (('single', 'single-step'), ('rollout', 'rollout'), ('persistence', 'persistence')):
            r[f'test RMSE {cn}'] = round(e[m]['active']['rmse'], 4)
        r['eval has count cols'] = 'tp_0.1' in e['single']['active']
    rows.append(r)

cols = ['fold', 'status', 'epochs done', 'current epoch / step', 'best epoch', 'best val', 'elapsed', 'ETA (all epochs)', 'log updated', 'GPU', 'sessions',
        'test RMSE single-step', 'test RMSE rollout', 'test RMSE persistence', 'eval has count cols']
df = pd.DataFrame(rows).reindex(columns=cols).fillna('')
print('refreshed (UTC):', datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S'), '  test RMSE = active cells, pooled over 10 test runs x 168 time steps')
display(df)
n_done = sum(1 for x in rows if x.get('status') == 'evaluated')
print(f'evaluated {n_done} / 6 folds' + ('; once all 6 folds are done and all have count columns, run code/44_exp2_summary.py to summarize' if n_done == 6 else ''))

## (3) (optional) show the last 15 lines of one fold's log

In [ ]:
FOLD = 'matthew'   # change to the fold you want to see
p = f'{OUT_ROOT}/exp2/fold_{FOLD}/train_stdout.log'
print(''.join(open(p, errors='replace').readlines()[-15:]) if os.path.exists(p) else 'no log')